In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import time
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.initializers import RandomNormal, GlorotUniform, HeNormal

# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Normalize input data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define initialization methods
initializers = {
    "Random": RandomNormal(mean=0.0, stddev=0.05),
    "Xavier": GlorotUniform(),
    "He": HeNormal()
}

results = []

# Train ANN using different initialization methods
for name, initializer in initializers.items():
    print("\nTraining with", name, "Initialization")

    # Create ANN model
    model = Sequential([
        Input(shape=(4,)),
        Dense(8, activation='relu', kernel_initializer=initializer),
        Dense(3, activation='softmax')
    ])

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Calculate training time
    start_time = time.time()

    # Train model
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=8,
        validation_data=(X_test, y_test),
        verbose=0
    )

    end_time = time.time()

    # Evaluate model
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    # Find convergence epoch (95% validation accuracy)
    convergence_epoch = None
    for epoch, acc in enumerate(history.history['val_accuracy'], start=1):
        if acc >= 0.95:
            convergence_epoch = epoch
            break

    # Store results
    results.append([
        name,
        round(accuracy * 100, 2),
        round(end_time - start_time, 2),
        convergence_epoch
    ])

    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Training Time:", round(end_time - start_time, 2), "seconds")
    print("Convergence Epoch:", convergence_epoch)

# Display final comparison
result_df = pd.DataFrame(
    results,
    columns=[
        "Initialization",
        "Accuracy (%)",
        "Training Time (seconds)",
        "Convergence Epoch"
    ]
)

print("\nFinal Comparison:")
print(result_df)


Training with Random Initialization
Accuracy: 96.67 %
Training Time: 8.41 seconds
Convergence Epoch: 37

Training with Xavier Initialization
Accuracy: 96.67 %
Training Time: 8.81 seconds
Convergence Epoch: 36

Training with He Initialization
Accuracy: 96.67 %
Training Time: 9.8 seconds
Convergence Epoch: 48

Final Comparison:
  Initialization  Accuracy (%)  Training Time (seconds)  Convergence Epoch
0         Random         96.67                     8.41                 37
1         Xavier         96.67                     8.81                 36
2             He         96.67                     9.80                 48
